# H-007 · Does Lottery Demand (MAX) Predict Lower Forward Returns?

Factor test for **H-007** (equities): whether stocks with extreme recent upside days (Bali, Cakici & Whitelaw 2011 MAX) earn lower next-week returns than peers (negative IC).

- **Idea** — Average of the `N` largest daily returns over the past `W` trading days (paper default **N=5**, **W=21**). High MAX = lottery-like recent path.
- **Claim** — High MAX predicts **lower** forward returns at 1d/5d/21d (negative IC).
- **Why it might work** — Preferential demand for lottery-like payoffs: retail / constrained investors overpay for stocks with extreme recent upside days; subsequent returns mean-revert.
- **Data** — Daily OHLCV long panel (`close`) from `s1_factor_panel_train.parquet` + SPY for idio-vol residuals via `add_idio_vol_factors`.

**No floor / no winsorize in the feature store:** `add_max_lottery_factors` does **not** floor returns and does **not** winsorize. Fewer than `N` finite returns in the `W`-bar window → NaN. If you need winsorization, apply it in §2 — not inside the factor API.

**This notebook screens** `N_EXTREMES × WINDOWS × MODES × NORMALIZE_OPTS` on research IS, plus **MAX residuals** vs idio-vol (no H-002 / realised-vol baselines). Store names do not encode `normalize`, so main columns are renamed to `max_lottery_{mode}_{N}_{W}_{raw|cs}`. Resid columns are `max_lottery_{mode}_resid_{N}_{W}` (always CS-rank residual of MAX on idio-vol; independent of the main-column normalize tag).

| Knob | Values |
|------|--------|
| `N_EXTREMES` | `[1, 5]` |
| `WINDOWS` | `[10, 21, 42]` (21 = Bali paper default) |
| `MODES` | `simple`, `log` |
| `NORMALIZE_OPTS` | `False` → tag `raw`; `True` → tag `cs` (**CS z-score** by date, not pct-rank) |
| Residuals | `add_residuals=True` after attaching `idio_vol` (SPY; `IDIO_WINDOW=20`) |
| Alphalens `periods` | `(1, 5, 21)` (primary narrative **5d**) |

**`mode` options:**

| Mode | Behavior |
|------|----------|
| `simple` (default) | Daily simple return `P_t / P_{t-1} - 1`, then mean of top `N` in `W`. |
| `log` | Daily log return `ln(P_t / P_{t-1})`, then mean of top `N` in `W`. |

Do **not** re-split the train parquet; do **not** use `s1_factor_panel_full.parquet` for window keep/kill.

**No H-002 / realised-vol baselines** in this notebook. Residuals are the nested control vs H-003 idio-vol.

Use `data.processing.s1_feature_store.add_max_lottery_factors` rather than reimplementing the factor inline.

Evaluation uses the S1 **trade-date** panel: Alphalens pivots `open` (no `shift(-1)`); labels are open-to-open. Prior close-to-close ICs are not comparable.


## 0. Imports & Config

Resolve the repo root; configure `N_EXTREMES`, `WINDOWS`, `MODES`, `NORMALIZE_OPTS`, `IDIO_WINDOW`, Alphalens `PERIODS`, and tearsheet paths. The train parquet is already research IS; do not calculate another cutoff here.


In [1]:
import os
import sys

import alphalens as al
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd

from data.ingestion.equity_fetcher import fetch_ohlcv
from data.processing.cleaner import forward_fill_panel
from data.processing.feature_implementation.beta_features import market_return_frame
from data.processing.s1_feature_store import add_idio_vol_factors, add_max_lottery_factors

# Jupyter cwd is often this notebook's folder, not the repo root; walk up until we find 01_data/ingestion.
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)

TEARSHEET_DIR = os.path.join(
    ROOT, "02_research", "notebooks", "s1_equities", "factor_tests", "tearsheets"
)

# --- Screen grids (hypothesis log) ---
N_EXTREMES = [1, 5]
WINDOWS = [10, 21, 42]
MODES = ["simple", "log"]  # feature_subset IDs

# --- Normalize screen (False → raw MAX; True → CS z-score) ---
NORMALIZE_OPTS = [False, True]

# --- Idio-vol control for residuals (H-003 default window) ---
IDIO_WINDOW = 20
BENCHMARK = "SPY"

# --- Fixed for this notebook ---
PERIODS = (1, 5, 21)  # S1 default; primary narrative = 5d
QUANTILES = 5
MAX_LOSS = 0.35


## 1. Data Loading

Load `s1_factor_panel_train.parquet` (daily OHLCV research IS) and SPY via `fetch_ohlcv` for idio-vol residuals. Confirm columns: `date`, `ticker`, `close`.

Do not apply another 70/30 split. Do not use `s1_factor_panel_full.parquet` for window keep/kill.


In [2]:
panel = pd.read_parquet(TRAIN_PANEL_PATH)
required = {"date", "ticker", "open", "close", "feature_date"}
missing = required - set(panel.columns)
if missing:
    raise ValueError(f"train panel missing columns: {sorted(missing)}")

panel = panel.copy()
panel["date"] = pd.to_datetime(panel["date"])

# Buffer past panel end so Alphalens can compute 21d forward returns near the last IS date.
start = panel["date"].min().strftime("%Y-%m-%d")
end = (panel["date"].max() + pd.Timedelta(days=40)).strftime("%Y-%m-%d")
spy = fetch_ohlcv(BENCHMARK, start, end)
market_returns = market_return_frame(spy)

print(
    f"rows={len(panel):,}  tickers={panel['ticker'].nunique():,}  "
    f"dates={panel['date'].nunique():,}  "
    f"[{panel['date'].min().date()} → {panel['date'].max().date()}]"
)
print(f"SPY market returns: {len(market_returns):,} rows")
panel.head()


rows=289,381  tickers=100  dates=2,915  [2010-01-05 → 2021-08-03]
SPY market returns: 2,942 rows


,date,ticker,open,high,low,close,volume,feature_date,fwd_ret_1,fwd_ret_5,fwd_ret_21
0,2010-01-05,AAPL,6.424143,6.421146,6.357683,6.406478,493729600.0,2010-01-04,-0.001025,-0.025210,-0.083271
1,2010-01-06,AAPL,6.417558,6.453779,6.383729,6.417557,601904800.0,2010-01-05,-0.012268,-0.030367,-0.101456
2,2010-01-07,AAPL,6.338825,6.443003,6.308892,6.315478,552160000.0,2010-01-06,-0.006848,-0.007745,-0.075844
3,2010-01-08,AAPL,6.295419,6.346309,6.258000,6.303801,477131200.0,2010-01-07,0.011888,0.002996,-0.066001
4,2010-01-11,AAPL,6.370258,6.346310,6.258300,6.345711,447610800.0,2010-01-08,-0.016965,-0.021005,-0.079464


## 2. Data Cleaning & Engineering

Forward-fill `close` via the project cleaner, then drop remaining nulls. Attach raw `idio_vol` (`normalize=False`) for residualization — `add_max_lottery_factors(..., add_residuals=True)` CS-ranks MAX and idio within date before OLS. The store itself applies **no** floor and **no** winsorize.

Point-in-time only: features at `t` use information available at or before `t`.


In [3]:
panel = forward_fill_panel(panel, columns=["close"], limit=5)
panel = panel.dropna(subset=["close"]).reset_index(drop=True)

panel = add_idio_vol_factors(
    panel,
    market_returns,
    feature_subset=["idio_vol"],
    windows=IDIO_WINDOW,
    normalize=False,
)
if "idio_vol" not in panel.columns:
    raise ValueError("expected idio_vol column after add_idio_vol_factors")

print(
    f"after clean: rows={len(panel):,}  "
    f"null close={panel['close'].isna().sum()}  "
    f"idio_vol finite={panel['idio_vol'].notna().sum():,}"
)


after clean: rows=289,381  null close=0  idio_vol finite=287,281


## 3. Modeling / Signal Construction

Call `add_max_lottery_factors` for each `(mode, normalize)` via `feature_subset=[mode]` with `n_extreme=N_EXTREMES` and `window=WINDOWS` (multi → `max_lottery_{mode}_{N}_{W}`). Rename mains to `max_lottery_{mode}_{N}_{W}_{raw|cs}` so both normalize settings coexist.

Residuals: call once per `mode` with `normalize=False`, `add_residuals=True` (resid does not depend on main-column normalize) and keep `max_lottery_{mode}_resid_{N}_{W}`.

Do not reimplement top-N extreme averaging inline.


In [4]:
FACTOR_COLS: list[str] = []
RESID_COLS: list[str] = []

for mode in MODES:
    for normalize in NORMALIZE_OPTS:
        tmp = add_max_lottery_factors(
            panel,
            n_extreme=N_EXTREMES,
            window=WINDOWS,
            feature_subset=[mode],
            normalize=normalize,
            add_residuals=False,
        )
        tag = "cs" if normalize else "raw"
        for n in N_EXTREMES:
            for w in WINDOWS:
                src = f"max_lottery_{mode}_{n}_{w}"
                dst = f"max_lottery_{mode}_{n}_{w}_{tag}"
                if src not in tmp.columns:
                    raise ValueError(
                        f"expected store column {src!r}, got {list(tmp.columns)}"
                    )
                panel[dst] = tmp[src]
                FACTOR_COLS.append(dst)

    # Residuals once per mode (CS-rank MAX ⊥ CS-rank idio; independent of normalize tag).
    tmp_r = add_max_lottery_factors(
        panel,
        n_extreme=N_EXTREMES,
        window=WINDOWS,
        feature_subset=[mode],
        normalize=False,
        add_residuals=True,
        idio_vol_col="idio_vol",
    )
    for n in N_EXTREMES:
        for w in WINDOWS:
            src = f"max_lottery_{mode}_resid_{n}_{w}"
            if src not in tmp_r.columns:
                raise ValueError(
                    f"expected resid column {src!r}, got {list(tmp_r.columns)}"
                )
            panel[src] = tmp_r[src]
            RESID_COLS.append(src)

ALL_FACTOR_COLS = FACTOR_COLS + RESID_COLS
print(f"MAX factors ({len(FACTOR_COLS)}):")
for c in FACTOR_COLS:
    print(f"  {c}")
print(f"MAX resid factors ({len(RESID_COLS)}):")
for c in RESID_COLS:
    print(f"  {c}")


MAX factors (24):
  max_lottery_simple_1_10_raw
  max_lottery_simple_1_21_raw
  max_lottery_simple_1_42_raw
  max_lottery_simple_5_10_raw
  max_lottery_simple_5_21_raw
  max_lottery_simple_5_42_raw
  max_lottery_simple_1_10_cs
  max_lottery_simple_1_21_cs
  max_lottery_simple_1_42_cs
  max_lottery_simple_5_10_cs
  max_lottery_simple_5_21_cs
  max_lottery_simple_5_42_cs
  max_lottery_log_1_10_raw
  max_lottery_log_1_21_raw
  max_lottery_log_1_42_raw
  max_lottery_log_5_10_raw
  max_lottery_log_5_21_raw
  max_lottery_log_5_42_raw
  max_lottery_log_1_10_cs
  max_lottery_log_1_21_cs
  max_lottery_log_1_42_cs
  max_lottery_log_5_10_cs
  max_lottery_log_5_21_cs
  max_lottery_log_5_42_cs
MAX resid factors (12):
  max_lottery_simple_resid_1_10
  max_lottery_simple_resid_1_21
  max_lottery_simple_resid_1_42
  max_lottery_simple_resid_5_10
  max_lottery_simple_resid_5_21
  max_lottery_simple_resid_5_42
  max_lottery_log_resid_1_10
  max_lottery_log_resid_1_21
  max_lottery_log_resid_1_42
  max_l

## 4. Evaluation

Alphalens IC + quintile spreads on research IS only at `periods=(1, 5, 21)` (primary narrative **5d**). Screen all main `N × W × mode × normalize` columns and residual columns. No H-002 / realised-vol baselines.


### 4.1 Window × N × mode × normalize screen

| Token | Meaning |
|-------|---------|
| **mode** | `simple` or `log` |
| **N** | Number of extreme returns (`N_EXTREMES`) |
| **W** | Trailing return window (`WINDOWS`) |
| **normalize** | `raw` = store `normalize=False`; `cs` = store `normalize=True` (**CS z-score**, not pct-rank) |
| **resid** | `max_lottery_{mode}_resid_{N}_{W}` = within-date OLS residual of CS-rank(MAX) on CS-rank(idio_vol) |

**Column patterns:** `max_lottery_{mode}_{N}_{W}_{raw|cs}` — e.g. `max_lottery_simple_5_21_cs`; resid `max_lottery_simple_resid_5_21`.

Primary ranking metric: **mean IC at 5d** (`ic_5d`). Expect **negative** IC for lottery demand. After the full table, a pivot compares `raw` vs `cs` for each `(mode, N, W)`.


In [5]:
def to_alphalens_prices(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide open matrix for Alphalens (trade-date panel; entry at open)."""
    prices = panel.pivot(index="date", columns="ticker", values="open")
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


def to_alphalens_factor(panel: pd.DataFrame, col: str) -> pd.Series:
    """MultiIndex (date, ticker) factor series for Alphalens."""
    factor = panel.set_index(["date", "ticker"])[col].dropna()
    factor.index = factor.index.set_levels(
        pd.to_datetime(factor.index.levels[0]), level=0
    )
    return factor.sort_index()


def parse_factor_name(col: str) -> dict:
    """Decode main ``max_lottery_{mode}_{N}_{W}_{raw|cs}`` or resid ``max_lottery_{mode}_resid_{N}_{W}``."""
    if not col.startswith("max_lottery_"):
        raise ValueError(f"unrecognized factor column: {col!r}")
    rest = col[len("max_lottery_"):]
    parts = rest.rsplit("_", 3)
    if len(parts) != 4:
        raise ValueError(f"unrecognized factor column: {col!r}")

    # Resid: {mode}_resid_{N}_{W} → parts = [mode, "resid", N, W]
    if parts[1] == "resid":
        mode, _, n_str, w_str = parts
        return {
            "kind": "resid",
            "mode": mode,
            "N": int(n_str),
            "W": int(w_str),
            "normalize": None,
            "norm_tag": "resid",
        }

    # Main: {mode}_{N}_{W}_{raw|cs}
    mode, n_str, w_str, tag = parts
    if tag not in {"raw", "cs"}:
        raise ValueError(f"expected normalize tag raw|cs, got {tag!r} in {col!r}")
    return {
        "kind": "main",
        "mode": mode,
        "N": int(n_str),
        "W": int(w_str),
        "normalize": tag == "cs",
        "norm_tag": tag,
    }


def _period_label(period_index: pd.Index, period: int, position: int):
    """Match Alphalens period label ('1D', '5D', …) or fall back by position."""
    for c in (f"{period}D", f"{period}d", period, str(period)):
        if c in period_index:
            return c
    return period_index[position]


def factor_screen_metrics(
    factor: pd.Series,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
) -> dict:
    """Mean IC and Q5−Q1 mean return spread for each forward period."""
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=factor,
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )
    mean_ic = al.performance.mean_information_coefficient(factor_data)
    mean_ret, _ = al.performance.mean_return_by_quantile(factor_data, demeaned=True)

    row = {}
    for i, p in enumerate(periods):
        ic_key = _period_label(mean_ic.index, p, i)
        ret_key = _period_label(mean_ret.columns, p, i)
        row[f"ic_{p}d"] = float(mean_ic.loc[ic_key])
        q_hi, q_lo = mean_ret.index.max(), mean_ret.index.min()
        row[f"spread_{p}d"] = float(
            mean_ret.loc[q_hi, ret_key] - mean_ret.loc[q_lo, ret_key]
        )
    return row


In [6]:
prices = to_alphalens_prices(panel)

In [7]:
rows = []
for col in ALL_FACTOR_COLS:
    meta = parse_factor_name(col)
    metrics = factor_screen_metrics(to_alphalens_factor(panel, col), prices)
    rows.append({"factor": col, **meta, **metrics})

summary = (
    pd.DataFrame(rows)
    .sort_values("ic_5d", ascending=True)  # lottery: more negative IC is better
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4f}".format)
summary


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

,factor,kind,mode,N,W,normalize,norm_tag,ic_1d,spread_1d,ic_5d,spread_5d,ic_21d,spread_21d
0,max_lottery_log_5_10_raw,main,log,5,10,False,raw,-0.0082,0.0000,-0.0042,0.0007,0.0012,0.0032
1,max_lottery_log_5_10_cs,main,log,5,10,True,cs,-0.0082,0.0000,-0.0042,0.0007,0.0012,0.0032
2,max_lottery_simple_5_10_cs,main,simple,5,10,True,cs,-0.0082,0.0001,-0.0042,0.0007,0.0012,0.0032
3,max_lottery_simple_5_10_raw,main,simple,5,10,False,raw,-0.0082,0.0001,-0.0042,0.0007,0.0012,0.0032
4,max_lottery_log_5_21_raw,main,log,5,21,False,raw,-0.0065,0.0002,-0.0019,0.0010,0.0047,0.0046
5,max_lottery_log_5_21_cs,main,log,5,21,True,cs,-0.0065,0.0002,-0.0019,0.0010,0.0047,0.0046
6,max_lottery_simple_5_21_cs,main,simple,5,21,True,cs,-0.0065,0.0003,-0.0019,0.0011,0.0048,0.0046
7,max_lottery_simple_5_21_raw,main,simple,5,21,False,raw,-0.0065,0.0003,-0.0019,0.0011,0.0048,0.0046
8,max_lottery_log_5_42_raw,main,log,5,42,False,raw,-0.0065,0.0003,-0.0015,0.0014,0.0020,0.0049
9,max_lottery_log_5_42_cs,main,log,5,42,True,cs,-0.0065,0.0003,-0.0015,0.0014,0.0020,0.0049


#### Normalize comparison (`raw` vs `cs` z-score)

Same `(mode, N, W)` side-by-side on **main** columns only: which `normalize` setting has more negative `ic_5d`? `delta_ic_5d = cs − raw` (negative ⇒ CS z-score more lottery-like / stronger negative IC).


In [8]:
main_summary = summary.loc[summary["kind"] == "main"].copy()
norm_cmp = (
    main_summary.pivot_table(
        index=["mode", "N", "W"],
        columns="norm_tag",
        values="ic_5d",
        aggfunc="first",
    )
    .reindex(columns=["raw", "cs"])
    .sort_index()
)
norm_cmp["delta_ic_5d"] = norm_cmp["cs"] - norm_cmp["raw"]
norm_cmp["winner"] = norm_cmp.apply(
    lambda r: (
        "cs"
        if r["cs"] < r["raw"]
        else ("raw" if r["raw"] < r["cs"] else "tie")
    ),
    axis=1,
)
print("normalize winners (more negative ic_5d = better for lottery):")
print(norm_cmp["winner"].value_counts().to_string())
norm_cmp


normalize winners (more negative ic_5d = better for lottery):
winner
tie    12


norm_tag        raw      cs  delta_ic_5d winner
mode   N W                                     
log    1 10 -0.0010 -0.0010       0.0000    tie
         21  0.0014  0.0014       0.0000    tie
         42  0.0019  0.0019       0.0000    tie
       5 10 -0.0042 -0.0042       0.0000    tie
         21 -0.0019 -0.0019       0.0000    tie
         42 -0.0015 -0.0015       0.0000    tie
simple 1 10 -0.0010 -0.0010       0.0000    tie
         21  0.0014  0.0014       0.0000    tie
         42  0.0019  0.0019       0.0000    tie
       5 10 -0.0042 -0.0042       0.0000    tie
         21 -0.0019 -0.0019       0.0000    tie
         42 -0.0014 -0.0014       0.0000    tie

#### Residuals vs main (nested control)

For each `(mode, N, W)`, compare `ic_5d` of the **cs** (z-scored) main column vs the resid column. Resid should retain incremental lottery signal after stripping CS idio-vol rank.


In [9]:
cs_main = main_summary.loc[main_summary["norm_tag"] == "cs", ["mode", "N", "W", "ic_5d"]].rename(
    columns={"ic_5d": "ic_5d_cs"}
)
resid_summary = summary.loc[summary["kind"] == "resid", ["mode", "N", "W", "ic_5d"]].rename(
    columns={"ic_5d": "ic_5d_resid"}
)
resid_cmp = cs_main.merge(resid_summary, on=["mode", "N", "W"], how="inner").sort_values(
    ["mode", "N", "W"]
)
resid_cmp["delta_ic_5d"] = resid_cmp["ic_5d_resid"] - resid_cmp["ic_5d_cs"]
resid_cmp


,mode,N,W,ic_5d_cs,ic_5d_resid,delta_ic_5d
7,log,1,10,-0.0010,0.0032,0.0042
9,log,1,21,0.0014,0.0074,0.0060
10,log,1,42,0.0019,0.0081,0.0062
0,log,5,10,-0.0042,-0.0001,0.0041
2,log,5,21,-0.0019,0.0017,0.0036
4,log,5,42,-0.0015,0.0038,0.0052
6,simple,1,10,-0.0010,0.0032,0.0042
8,simple,1,21,0.0014,0.0074,0.0060
11,simple,1,42,0.0019,0.0081,0.0062
1,simple,5,10,-0.0042,-0.0002,0.0040


### 4.2 Full tear sheet (manual pick)

Review §4.1, then set `TEAR_KIND`, `TEAR_MODE`, `TEAR_N`, `TEAR_WINDOW`, and `TEAR_NORMALIZE` below. Nothing is auto-selected from the winner.

The tear is displayed in-notebook **and** saved as a multi-page PDF under `02_research/notebooks/s1_equities/factor_tests/tearsheets/` named `H-007_{factor_col}.pdf`. Re-running overwrites the same path.


In [10]:
def max_lottery_factor_col(
    kind: str,
    mode: str,
    n_extreme: int,
    window: int,
    normalize: bool = True,
) -> str:
    """Column name for a screened MAX / MAX-resid factor."""
    if kind == "resid":
        return f"max_lottery_{mode}_resid_{n_extreme}_{window}"
    if kind != "main":
        raise ValueError(f"kind must be 'main' or 'resid', got {kind!r}")
    tag = "cs" if normalize else "raw"
    return f"max_lottery_{mode}_{n_extreme}_{window}_{tag}"


def run_full_tear(
    panel: pd.DataFrame,
    factor_col: str,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
    tearsheet_dir: str = TEARSHEET_DIR,
):
    """Build factor_data, run Alphalens full tear, save figs to multi-page PDF.

    Alphalens calls plt.show() after each plot, which clears figures under Agg.
    Temporarily replace plt.show so each figure is written into the PDF before close.
    """
    if factor_col not in panel.columns:
        raise ValueError(
            f"{factor_col!r} not in panel — pick kind/mode/N/W/normalize that were screened "
            f"(available: {ALL_FACTOR_COLS})"
        )
    plt.close("all")
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=to_alphalens_factor(panel, factor_col),
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )

    os.makedirs(tearsheet_dir, exist_ok=True)
    out_path = os.path.join(tearsheet_dir, f"H-007_{factor_col}.pdf")
    pdf = PdfPages(out_path)
    n_pages = 0
    _original_show = plt.show

    def _show_and_savefig(*args, **kwargs):
        nonlocal n_pages
        for num in list(plt.get_fignums()):
            fig = plt.figure(num)
            if fig.axes:
                pdf.savefig(fig, bbox_inches="tight")
                n_pages += 1
        plt.close("all")

    plt.show = _show_and_savefig
    try:
        al.tears.create_full_tear_sheet(factor_data, long_short=True)
        _show_and_savefig()
    finally:
        plt.show = _original_show
        pdf.close()
        plt.close("all")

    print(f"Wrote {out_path} ({n_pages} pages)")
    return factor_data


In [11]:
# Review §4.1, then edit these (nothing auto-selected).
TEAR_KIND = "main"       # main | resid
TEAR_MODE = "log"        # simple | log
TEAR_N = 5               # paper default
TEAR_WINDOW = 10         # paper default until you change after the screen
TEAR_NORMALIZE = True    # False → raw; True → cs (z-score); ignored when kind=resid

tear_col = max_lottery_factor_col(
    TEAR_KIND, TEAR_MODE, TEAR_N, TEAR_WINDOW, TEAR_NORMALIZE
)
print(f"Tear sheet factor: {tear_col}")
tear_data = run_full_tear(panel, tear_col, prices)


Tear sheet factor: max_lottery_log_5_10_cs


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Quantiles Statistics


,min,max,mean,std,count,count %
factor_quantile,,,,,,
1,-6.2011,-0.4316,-1.0279,0.2989,57680,20.1437
2,-1.0307,-0.0664,-0.5464,0.1292,57028,19.9160
3,-0.5927,0.3666,-0.1849,0.1327,56928,19.8811
4,-0.2415,1.0897,0.2611,0.2065,57028,19.9160
5,0.0261,8.6131,1.4925,1.0337,57679,20.1433


Returns Analysis


...

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\performance.py:118: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  grouper.append(pd.Grouper(freq=by_time))
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:928: UserWarning: Skipping return periods that aren't exact multiples of days.
  warnings.warn(


Turnover Analysis


,1D,5D,21D
Quantile 1 Mean Turnover,0.2280,0.5230,0.7130
Quantile 2 Mean Turnover,0.4260,0.6910,0.7670
Quantile 3 Mean Turnover,0.4430,0.7140,0.7840
Quantile 4 Mean Turnover,0.3770,0.6710,0.7720
Quantile 5 Mean Turnover,0.1710,0.4220,0.6220


,1D,5D,21D
Mean Factor Rank Autocorrelation,0.9030,0.5890,0.2560


Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\02_research\notebooks\factor_tests\tearsheets\H-007_max_lottery_log_5_10_cs.pdf (3 pages)


## Conclusion

Fill after running the screen (edit this cell with your notes):

- **Best `ic_5d` combo (most negative):** mode / N / W / normalize = …
- **Normalize:** does `cs` (z-score) or `raw` win more often in the §4.1 pivot? (and at 1d / 21d if materially different)
- **Paper default `N=5`, `W=21`:** hold up vs other grid points?
- **Residuals:** does `MAX_resid` retain negative IC after stripping idio-vol? Keep / drop resid.
- **Keep / kill (H-007 alone):** …
- **Deferred:** horse-race vs H-002 GK / realised vol in a later pass if keeping.
